In [ ]:
from kafka import KafkaConsumer
import pandas as pd
import pickle

# Function to load the model and vectorizer
def load_model_and_vectorizer():
    try:
        with open('logistic_regression_model.pkl', 'rb') as model_file:
            model = pickle.load(model_file)

        with open('vectorizer.pkl', 'rb') as vectorizer_file:
            vectorizer = pickle.load(vectorizer_file)

        return model, vectorizer
    except Exception as e:
        print(f"Error loading model or vectorizer: {e}")
        raise

# Load pre-trained model and vectorizer
model, vectorizer = load_model_and_vectorizer()

# Initialize Kafka Consumer
consumer = KafkaConsumer(
    'test',
    bootstrap_servers='localhost:9092',
    value_deserializer=lambda v: v.decode('utf-8')  # Deserialize plain string
)

# Initialize a DataFrame to keep count of the categories
count_df = pd.DataFrame({'category': ['politics', 'tech', 'sports', 'entertainment', 'business'], 'count': [0, 0, 0, 0, 0]})

# Function to update counts
def update_counts(predicted_category):
    count_df.loc[count_df['category'] == predicted_category, 'count'] += 1
    # Save the updated count to a CSV file
    count_df.to_csv('category_trend.csv', index=False)

# Process messages from Kafka and classify them
def classify_messages():
    for message in consumer:
        if message.value:  # Check if the message value is not empty
            try:
                text = message.value
                # Vectorize the incoming text
                text_vectorized = vectorizer.transform([text])
                # Predict the category
                predicted_category = model.predict(text_vectorized)[0]
                print(f"Text: {text} | Predicted category: {predicted_category}")
                # Update the counts and save to CSV
                update_counts(predicted_category)
            except Exception as e:
                print(f"Error processing message: {e}, value: {message.value}")
        else:
            print("Received empty message.")

# Run the consumer and classification
if __name__ == "__main__":
    classify_messages()
